# DepMamba-X + EvoAttack-X + RARE + SAGE IDS — Full Implementation

This notebook implements the framework described in **Scopus VS_93.docx**: multi-IDS traffic integration, robust feature engineering, DepMamba-X, EvoAttack-X, RARE, SAGE, U-CAGE, LATe and X-EASE.

The paper specifies the Dataset for Detection in Multi-IDS Environment and the Kaggle source in Section 5. fileciteturn0file0L242-L255

**Important:** the notebook trains on the real dataset supplied by the user. It does **not** hard-code paper performance values. Any reported metric is produced from the loaded data and trained model.

In [ ]:
# 1. Imports and reproducibility
import os, glob, random, warnings, time, math, json, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except ImportError as e:
    raise ImportError("Install PyTorch first: pip install torch") from e

torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)


In [ ]:
# 2. Configuration
DATA_DIR="/content/dataset"       # <-- change to your extracted Kaggle dataset folder
CSV_PATH=None                     # e.g. "/content/dataset/IDS1.csv"; None = discover CSVs
SAVE_DIR="/content/DepMambaX_results"
os.makedirs(SAVE_DIR,exist_ok=True)

TARGET_CANDIDATES=[
    "label","Label","target","Target","class","Class","attack","Attack",
    "is_attack","is_attacked","erroris_attack","malicious","Malicious Packet"
]

SEQ_LEN=16
MAX_FEATURES=32
HIDDEN=64
EMBED_DIM=64
NUM_EXPERTS=4
TOP_K=2
BATCH_SIZE=128
PRETRAIN_EPOCHS=5
RARE_EPOCHS=5
ATTACK_STEPS=3
ATTACK_EPS=0.05
ATTACK_ALPHA=0.02
LR=1e-3

print("Save directory:",SAVE_DIR)


In [ ]:
# 3. Locate and inspect the dataset
if CSV_PATH is not None:
    csv_files=[CSV_PATH]
else:
    csv_files=sorted(glob.glob(os.path.join(DATA_DIR,"**","*.csv"),recursive=True))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found under {DATA_DIR}. Download/extract the Kaggle Dataset for Detection in Multi-IDS Environment "
        "and set DATA_DIR or CSV_PATH."
    )

print("CSV files found:",len(csv_files))
for f in csv_files[:20]:
    print(f)

frames=[]
for f in csv_files:
    try:
        d=pd.read_csv(f,low_memory=False)
        d["__source_file__"]=os.path.basename(f)
        frames.append(d)
        print(os.path.basename(f),d.shape)
    except Exception as e:
        print("Skipped:",f,"->",e)

if not frames:
    raise RuntimeError("No readable CSV files.")
raw=pd.concat(frames,ignore_index=True)
print("Combined shape:",raw.shape)
display(raw.head())


In [ ]:
# 4. Target-column detection and label harmonization
def find_target(df):
    for c in TARGET_CANDIDATES:
        if c in df.columns: return c
    # fallback: common binary/multiclass names
    for c in df.columns:
        s=df[c].dropna()
        if 1 < s.nunique() <= 50 and any(k in c.lower() for k in ["label","attack","class","target","malicious"]):
            return c
    raise ValueError("Target column not detected. Set TARGET_COLUMN manually.")

TARGET_COLUMN=find_target(raw)
print("Target:",TARGET_COLUMN)
print(raw[TARGET_COLUMN].value_counts(dropna=False).head(30))

# Convert labels to compact integer IDs.
raw=raw.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
label_text=raw[TARGET_COLUMN].astype(str).str.strip()
classes=sorted(label_text.unique().tolist())
label_to_id={v:i for i,v in enumerate(classes)}
raw["__y__"]=label_text.map(label_to_id).astype(int)
print("Classes:",len(classes),classes[:30])


In [ ]:
# 5. Robust traffic feature engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer

DROP_ALWAYS={"__y__",TARGET_COLUMN,"__source_file__"}
# Raw IP/payload strings are useful source fields but are not directly numeric model inputs.
# Encode selected string columns using deterministic hashes when they contain useful traffic identity information.
df=raw.copy()

def hash_numeric(s,bins=2048):
    return s.astype(str).map(lambda x: (hash(x) % bins)/float(bins)).astype(np.float32)

numeric_cols=df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols=[c for c in numeric_cols if c not in DROP_ALWAYS]

# Add deterministic numeric representations for common packet fields if present.
for c in ["src_ip","dst_ip","protocol","tcp_payload_hex","tcp_payload_ascii"]:
    if c in df.columns:
        newc=f"__hash_{c}"
        df[newc]=hash_numeric(df[c])
        feature_cols.append(newc)

feature_cols=list(dict.fromkeys(feature_cols))
if not feature_cols:
    raise ValueError("No usable numeric traffic features were found.")

X=df[feature_cols].replace([np.inf,-np.inf],np.nan)
y=df["__y__"].values

# Split before fitting imputer/scaler/MI to avoid train-test leakage.
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=SEED,stratify=y
)

imp=SimpleImputer(strategy="median")
X_train_i=imp.fit_transform(X_train)
X_test_i=imp.transform(X_test)

scaler=RobustScaler()
X_train_s=scaler.fit_transform(X_train_i)
X_test_s=scaler.transform(X_test_i)

# Mutual-information feature selection, as specified by the paper.
mi=mutual_info_classif(X_train_s,y_train,random_state=SEED)
rank=np.argsort(mi)[::-1]
k=min(MAX_FEATURES,X_train_s.shape[1])
sel_idx=rank[:k]
selected_features=[feature_cols[i] for i in sel_idx]
X_train_s=X_train_s[:,sel_idx].astype(np.float32)
X_test_s=X_test_s[:,sel_idx].astype(np.float32)

print("Original features:",len(feature_cols))
print("Selected features:",len(selected_features))
print(selected_features)


In [ ]:
# 6. Sequence construction
# Records are kept in their original CSV-concatenated order, matching the paper's causal traffic-window description.
def make_sequences(X,y,seq_len):
    n=len(X)
    if n<seq_len: raise ValueError("Not enough records for SEQ_LEN.")
    xs=[]; ys=[]
    for i in range(seq_len-1,n):
        xs.append(X[i-seq_len+1:i+1])
        ys.append(y[i])
    return np.asarray(xs,dtype=np.float32),np.asarray(ys,dtype=np.int64)

Xtr_seq,ytr_seq=make_sequences(X_train_s,y_train,SEQ_LEN)
Xte_seq,yte_seq=make_sequences(X_test_s,y_test,SEQ_LEN)

print("Train sequences:",Xtr_seq.shape)
print("Test sequences :",Xte_seq.shape)

class TrafficDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]

train_loader=DataLoader(TrafficDataset(Xtr_seq,ytr_seq),batch_size=BATCH_SIZE,shuffle=True,drop_last=False)
test_loader=DataLoader(TrafficDataset(Xte_seq,yte_seq),batch_size=BATCH_SIZE,shuffle=False)
NUM_CLASSES=len(classes)


## 7. DepMamba-X

The paper describes DepMamba-X as a causal long-range state-space representation with a residual pathway and causal depthwise-separable temporal projection. fileciteturn0file0L311-L320

The implementation below uses a compact gated state-space recurrence as a dependency-free Mamba-style fallback. If `mamba-ssm` is installed, it can be substituted, but the notebook remains runnable without that external package.

In [ ]:
# 7. DepMamba-X implementation
class GatedStateSpace(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.in_proj=nn.Linear(dim,2*dim)
        self.state_proj=nn.Linear(dim,dim)
        self.gate=nn.Linear(dim,dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self,x):
        # causal recurrent state: h_t = g_t*h_{t-1} + (1-g_t)*candidate_t
        u,v=self.in_proj(x).chunk(2,dim=-1)
        h=torch.zeros_like(u[:,0])
        outs=[]
        for t in range(x.size(1)):
            g=torch.sigmoid(self.gate(u[:,t]))
            cand=torch.tanh(self.state_proj(v[:,t]))
            h=g*h+(1-g)*cand
            outs.append(h.unsqueeze(1))
        return self.norm(torch.cat(outs,dim=1))

class CausalDepthwiseProjection(nn.Module):
    def __init__(self,dim,kernel=3):
        super().__init__()
        self.dw=nn.Conv1d(dim,dim,kernel,padding=kernel-1,groups=dim)
        self.pw=nn.Conv1d(dim,dim,1)
        self.norm=nn.LayerNorm(dim)
    def forward(self,x):
        z=x.transpose(1,2)
        z=self.dw(z)[...,:x.size(1)]
        z=self.pw(z).transpose(1,2)
        return self.norm(F.gelu(z))

class DepMambaX(nn.Module):
    def __init__(self,in_dim,hidden=64,embed_dim=64):
        super().__init__()
        self.input_proj=nn.Linear(in_dim,hidden)
        self.ssm=GatedStateSpace(hidden)
        self.local=CausalDepthwiseProjection(hidden)
        self.out=nn.Sequential(nn.Linear(hidden,embed_dim),nn.LayerNorm(embed_dim))
    def forward(self,x):
        z=self.input_proj(x)
        h=self.ssm(z)
        h=h+z
        h=self.local(h)+h
        pooled=h.mean(dim=1)
        return self.out(pooled)


## 8. SAGE classifier

The paper specifies a differentiable soft-routing stage for optimization and hard Top-K routing during inference. fileciteturn0file0L590-L611

In [ ]:
class SAGE(nn.Module):
    def __init__(self,dim,num_classes,num_experts=4,top_k=2):
        super().__init__()
        self.num_experts=num_experts; self.top_k=top_k
        self.gate=nn.Linear(dim,num_experts)
        self.experts=nn.ModuleList([
            nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,num_classes))
            for _ in range(num_experts)
        ])
    def forward(self,x,hard=False):
        logits=self.gate(x)
        if hard:
            idx=torch.topk(logits,self.top_k,dim=-1).indices
            w=torch.zeros_like(logits).scatter(1,idx,1.0)
            w=w/(w.sum(1,keepdim=True)+1e-8)
        else:
            w=F.softmax(logits,dim=-1)
        out=torch.stack([e(x) for e in self.experts],dim=1)
        return (out*w.unsqueeze(-1)).sum(1),logits

class FullModel(nn.Module):
    def __init__(self,in_dim,num_classes):
        super().__init__()
        self.encoder=DepMambaX(in_dim,HIDDEN,EMBED_DIM)
        self.refine=nn.Sequential(nn.Linear(EMBED_DIM,EMBED_DIM),nn.GELU(),nn.LayerNorm(EMBED_DIM))
        self.sage=SAGE(EMBED_DIM,num_classes,NUM_EXPERTS,TOP_K)
    def encode(self,x): return self.encoder(x)
    def forward(self,x,hard=False):
        z=self.refine(self.encoder(x))
        return self.sage(z,hard=hard)[0]

model=FullModel(Xtr_seq.shape[-1],NUM_CLASSES).to(device)
print(model)
print("Parameters:",sum(p.numel() for p in model.parameters()))


In [ ]:
# 9. Clean pretraining of DepMamba-X + SAGE
criterion=nn.CrossEntropyLoss()
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)

history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}

def accuracy_from_logits(logits,y):
    return (logits.argmax(1)==y).float().mean().item()

for epoch in range(PRETRAIN_EPOCHS):
    model.train(); losses=[]; accs=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad()
        logits=model(xb)
        loss=criterion(logits,yb)
        loss.backward(); opt.step()
        losses.append(loss.item()); accs.append(accuracy_from_logits(logits,yb))
    model.eval(); vl=[]; va=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb,yb=xb.to(device),yb.to(device)
            lg=model(xb)
            vl.append(criterion(lg,yb).item()); va.append(accuracy_from_logits(lg,yb))
    history["train_loss"].append(np.mean(losses)); history["train_acc"].append(np.mean(accs))
    history["val_loss"].append(np.mean(vl)); history["val_acc"].append(np.mean(va))
    print(f"Epoch {epoch+1}/{PRETRAIN_EPOCHS} | loss {history['train_loss'][-1]:.4f} | acc {history['train_acc'][-1]:.4f} | val {history['val_acc'][-1]:.4f}")

torch.save(model.state_dict(),os.path.join(SAVE_DIR,"clean_pretrained_model.pt"))


## 10. EvoAttack-X

The paper specifies constrained PGD perturbations, a heterogeneous attack pool, evolutionary fitness, crossover/mutation and strongest-attack selection. fileciteturn0file0L393-L401

For reproducibility and memory efficiency, the implementation generates adversarial batches on demand rather than materializing an entire adversarial dataset.

In [ ]:
# 10. EvoAttack-X: constrained PGD + heterogeneous candidate pool + evolutionary selection
def project_delta(delta,eps):
    return torch.clamp(delta,-eps,eps)

def pgd_attack(model,x,y,eps,alpha,steps):
    was_training=model.training
    model.eval()
    x0=x.detach()
    delta=torch.empty_like(x0).uniform_(-eps,eps)
    for _ in range(steps):
        adv=(x0+delta).detach().requires_grad_(True)
        loss=F.cross_entropy(model(adv),y)
        grad=torch.autograd.grad(loss,adv,retain_graph=False,create_graph=False)[0]
        delta=project_delta(delta+alpha*grad.sign(),eps)
    adv=(x0+delta).detach()
    if was_training: model.train()
    return adv

ATTACK_POOL=[
    {"eps":0.02,"alpha":0.008,"steps":2},
    {"eps":0.04,"alpha":0.012,"steps":3},
    {"eps":0.06,"alpha":0.015,"steps":4},
    {"eps":0.08,"alpha":0.020,"steps":4},
]

def attack_fitness(clean_logits,adv_logits,delta,eps):
    pc=F.softmax(clean_logits,dim=1).max(1).values
    pa=F.softmax(adv_logits,dim=1)
    true_conf=pa.gather(1,clean_logits.argmax(1,keepdim=True)).squeeze(1)
    degradation=(pc-true_conf).mean()
    effectiveness=(adv_logits.argmax(1)!=clean_logits.argmax(1)).float().mean()
    magnitude=delta.abs().mean()
    return (effectiveness+degradation-0.1*magnitude/(eps+1e-8)).item()

def evolutionary_adversarial(model,x,y):
    clean_logits=model(x)
    candidates=[]
    for cfg in ATTACK_POOL:
        adv=pgd_attack(model,x,y,cfg["eps"],cfg["alpha"],cfg["steps"])
        adv_logits=model(adv)
        fit=attack_fitness(clean_logits,adv_logits,adv-x,cfg["eps"])
        candidates.append((fit,adv,cfg))
    candidates.sort(key=lambda z:z[0],reverse=True)
    return candidates[0][1].detach(),candidates[0][2],candidates[0][0]


## 11. RARE

RARE jointly uses clean classification, clean/adversarial consistency and taxonomy-aware contrastive structuring. The paper explicitly freezes DepMamba-X during RARE optimization. fileciteturn0file0L486-L515

In [ ]:
# 11. RARE loss and training
class ContrastiveHead(nn.Module):
    def __init__(self,dim=EMBED_DIM,proj=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,proj))
    def forward(self,z):
        return F.normalize(self.net(z),dim=-1)

contrastive_head=ContrastiveHead().to(device)

# Freeze DepMamba-X as described for RARE.
for p in model.encoder.parameters(): p.requires_grad=False

rare_params=[p for p in model.parameters() if p.requires_grad]+list(contrastive_head.parameters())
rare_opt=torch.optim.AdamW(rare_params,lr=LR,weight_decay=1e-4)

def rare_contrastive(z1,z2,y,temp=0.2):
    p1=contrastive_head(z1); p2=contrastive_head(z2)
    z=torch.cat([p1,p2],0)
    yy=torch.cat([y,y],0)
    sim=z@z.T/temp
    eye=torch.eye(len(z),device=z.device).bool()
    sim=sim.masked_fill(eye,-1e9)
    same=(yy[:,None]==yy[None,:]) & (~eye)
    # Multi-positive supervised contrastive objective.
    logp=sim-F.logsumexp(sim,dim=1,keepdim=True)
    pos_count=same.sum(1).clamp_min(1)
    return -(logp*same).sum(1).div(pos_count).mean()

def consistency_loss(a,b):
    return F.kl_div(F.log_softmax(a,1),F.softmax(b.detach(),1),reduction="batchmean")

rare_history=[]
for epoch in range(RARE_EPOCHS):
    model.train(); total=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        adv,_,_=evolutionary_adversarial(model,xb,yb)
        zc=model.encode(xb).detach()
        za=model.encode(adv).detach()
        rc=model.refine(zc); ra=model.refine(za)
        lc=model.sage(rc)[0]; la=model.sage(ra)[0]
        loss_cls=criterion(lc,yb)
        loss_cons=consistency_loss(lc,la)
        loss_ctr=rare_contrastive(zc,za,yb)
        loss=loss_cls+0.5*loss_cons+0.2*loss_ctr
        rare_opt.zero_grad(); loss.backward(); rare_opt.step()
        total+=loss.item()
    rare_history.append(total/len(train_loader))
    print(f"RARE epoch {epoch+1}/{RARE_EPOCHS} loss={rare_history[-1]:.4f}")

torch.save({"model":model.state_dict(),"contrastive":contrastive_head.state_dict()},
           os.path.join(SAVE_DIR,"DepMambaX_EvoAttackX_RARE.pt"))


## 12. U-CAGE

U-CAGE combines evidential uncertainty and an energy score, with thresholds calibrated on validation data. fileciteturn0file0L613-L633

In [ ]:
# 12. U-CAGE uncertainty and OOD scoring
@torch.no_grad()
def ucage_scores(logits,energy_temp=1.0):
    evidence=F.softplus(logits)
    alpha=evidence+1.0
    S=alpha.sum(1)
    uncertainty=len(classes)/S
    energy=-energy_temp*torch.logsumexp(logits/energy_temp,dim=1)
    return uncertainty,energy

# Collect validation-like scores from the held-out set for threshold calibration.
model.eval()
all_logits=[]; all_y=[]
with torch.no_grad():
    for xb,yb in test_loader:
        all_logits.append(model(xb.to(device)).cpu())
        all_y.append(yb)
all_logits=torch.cat(all_logits).to(device)
all_y=torch.cat(all_y).to(device)
unc,energy=ucage_scores(all_logits)
u_threshold=float(torch.quantile(unc,0.95))
e_threshold=float(torch.quantile(energy,0.95))
print("U-CAGE uncertainty threshold:",u_threshold)
print("U-CAGE energy threshold:",e_threshold)


## 13. LATe

LATe freezes DepMamba-X and adapts selected downstream layers through low-rank LoRA parameters while retaining historical replay samples. fileciteturn0file0L634-L656

In [ ]:
# 13. Lightweight LoRA adapter for continual adaptation
class LoRALinear(nn.Module):
    def __init__(self,base,rank=4,alpha=8):
        super().__init__()
        self.base=base
        for p in self.base.parameters(): p.requires_grad=False
        self.A=nn.Parameter(torch.randn(rank,base.in_features)*0.01)
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        self.scale=alpha/rank
    def forward(self,x):
        return self.base(x)+self.scale*(x@self.A.T@self.B.T)

def add_lora_to_sage(model,rank=4):
    for e in model.sage.experts:
        if isinstance(e[0],nn.Linear):
            e[0]=LoRALinear(e[0],rank).to(device)
    model.sage.gate=LoRALinear(model.sage.gate,rank).to(device)

def late_adapt(model,new_loader,replay_loader=None,epochs=2,lr=5e-4):
    add_lora_to_sage(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=lr)
    model.train()
    for ep in range(epochs):
        for xb,yb in new_loader:
            xb,yb=xb.to(device),yb.to(device)
            loss=criterion(model(xb),yb)
            if replay_loader is not None:
                try:
                    xr,yr=next(replay_it)
                except:
                    replay_it=iter(replay_loader); xr,yr=next(replay_it)
                loss=loss+0.3*criterion(model(xr.to(device)),yr.to(device))
            opt.zero_grad();loss.backward();opt.step()
    return model

# Usage example after a confirmed emerging-attack loader is prepared:
# model=late_adapt(model,new_attack_loader,train_loader,epochs=2)


## 14. Evaluation: Accuracy, Precision, Recall, F1, MCC, AUC

These are generated from the actual trained model. The paper requests comparative, adversarial, K-fold and statistical analyses. fileciteturn0file0L697-L718

In [ ]:
# 14. Evaluation utilities
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,matthews_corrcoef,roc_auc_score,confusion_matrix
from sklearn.preprocessing import label_binarize

@torch.no_grad()
def predict_loader(model,loader):
    model.eval(); ys=[]; ps=[]; probs=[]
    for xb,yb in loader:
        lg=model(xb.to(device),hard=True)
        pr=F.softmax(lg,1).cpu().numpy()
        probs.append(pr); ps.append(pr.argmax(1)); ys.append(yb.numpy())
    return np.concatenate(ys),np.concatenate(ps),np.concatenate(probs)

y_true,y_pred,y_prob=predict_loader(model,test_loader)
avg="binary" if NUM_CLASSES==2 else "weighted"
metrics={
    "Accuracy":accuracy_score(y_true,y_pred),
    "Precision":precision_score(y_true,y_pred,average=avg,zero_division=0),
    "Recall":recall_score(y_true,y_pred,average=avg,zero_division=0),
    "F1":f1_score(y_true,y_pred,average=avg,zero_division=0),
    "MCC":matthews_corrcoef(y_true,y_pred)
}
if NUM_CLASSES==2:
    metrics["AUC"]=roc_auc_score(y_true,y_prob[:,1])
else:
    metrics["AUC"]=roc_auc_score(label_binarize(y_true,classes=np.arange(NUM_CLASSES)),y_prob,average="weighted",multi_class="ovr")
print(pd.Series(metrics))
pd.DataFrame([metrics]).to_excel(os.path.join(SAVE_DIR,"proposed_metrics.xlsx"),index=False)


In [ ]:
# 15. Confusion matrix
cm=confusion_matrix(y_true,y_pred)
fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title("DepMamba-X + EvoAttack-X + RARE — Confusion Matrix")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(classes,rotation=45,ha="right"); ax.set_yticklabels(classes)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR,"confusion_matrix.png"),dpi=600,bbox_inches="tight")
plt.show()
pd.DataFrame(cm,index=classes,columns=classes).to_excel(os.path.join(SAVE_DIR,"confusion_matrix.xlsx"))


In [ ]:
# 16. Training history
h=pd.DataFrame(history)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(h.index+1,h.train_acc,marker="o",label="Train Accuracy")
ax.plot(h.index+1,h.val_acc,marker="o",label="Validation Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Training and Validation Accuracy")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR,"training_accuracy.png"),dpi=600,bbox_inches="tight"); plt.show()

fig,ax=plt.subplots(figsize=(8,5))
ax.plot(h.index+1,h.train_loss,marker="o",label="Train Loss")
ax.plot(h.index+1,h.val_loss,marker="o",label="Validation Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR,"training_loss.png"),dpi=600,bbox_inches="tight"); plt.show()
h.to_excel(os.path.join(SAVE_DIR,"model_history.xlsx"),index=False)


## 17. SHAP explanation

X-EASE uses SHAP feature attribution and counterfactual analysis. fileciteturn0file0L657-L679

The SHAP cell explains a compact wrapper around the frozen traffic representation. It uses a background sample from the training set and explains the final classifier.

In [ ]:
# 17. SHAP explanation
# If SHAP is unavailable: pip install shap
try:
    import shap
except ImportError:
    print("Install SHAP with: pip install shap")
    shap=None

class NumpyPredictor:
    def __init__(self,model,seq_len,n_features):
        self.model=model; self.seq_len=seq_len; self.n_features=n_features
    def __call__(self,X):
        X=np.asarray(X,dtype=np.float32).reshape(-1,self.seq_len,self.n_features)
        with torch.no_grad():
            lg=self.model(torch.tensor(X).to(device))
            return F.softmax(lg,1).cpu().numpy()

if shap is not None:
    bg=Xtr_seq[:min(50,len(Xtr_seq))].reshape(min(50,len(Xtr_seq)),-1)
    ex=Xte_seq[:min(20,len(Xte_seq))].reshape(min(20,len(Xte_seq)),-1)
    pred=NumpyPredictor(model,SEQ_LEN,Xtr_seq.shape[-1])
    explainer=shap.KernelExplainer(pred,bg)
    sv=explainer.shap_values(ex,nsamples=100)
    # Aggregate absolute attribution over temporal positions.
    if isinstance(sv,list):
        vals=np.mean(np.abs(np.stack(sv)),axis=0)
    else:
        vals=np.mean(np.abs(np.asarray(sv)),axis=tuple(range(0,np.asarray(sv).ndim-2)))
    vals=np.asarray(vals).reshape(len(ex),SEQ_LEN,Xtr_seq.shape[-1]).mean(axis=1).mean(axis=0)
    order=np.argsort(vals)[::-1][:min(10,len(selected_features))]
    plt.figure(figsize=(8,5))
    plt.barh(np.array(selected_features)[order][::-1],vals[order][::-1])
    plt.xlabel("Mean |SHAP value|"); plt.title("X-EASE — SHAP Feature Importance")
    plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR,"shap_feature_importance.png"),dpi=600,bbox_inches="tight"); plt.show()
    pd.DataFrame({"feature":np.array(selected_features)[order],"mean_abs_shap":vals[order]}).to_excel(
        os.path.join(SAVE_DIR,"shap_values.xlsx"),index=False)


## 18. LIME explanation

In [ ]:
# 18. LIME explanation
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    print("Install LIME with: pip install lime")
    LimeTabularExplainer=None

if LimeTabularExplainer is not None:
    bg=Xtr_seq.reshape(len(Xtr_seq),-1)
    ex=Xte_seq[0].reshape(1,-1)
    explainer=LimeTabularExplainer(
        bg,feature_names=[f"{f}_t{t}" for t in range(SEQ_LEN) for f in selected_features],
        class_names=classes,mode="classification",random_state=SEED
    )
    explanation=explainer.explain_instance(ex[0],NumpyPredictor(model,SEQ_LEN,Xtr_seq.shape[-1]),num_features=min(12,bg.shape[1]))
    fig=explanation.as_pyplot_figure()
    fig.tight_layout(); fig.savefig(os.path.join(SAVE_DIR,"lime_explanation.png"),dpi=600,bbox_inches="tight")
    plt.show()
    pd.DataFrame(explanation.as_list(),columns=["feature","weight"]).to_excel(os.path.join(SAVE_DIR,"lime_values.xlsx"),index=False)


## 19. Adversarial robustness analysis

The paper evaluates clean accuracy/F1, adversarial accuracy/F1, attack success rate and robustness retention. fileciteturn0file0L772-L799

In [ ]:
# 19. Adversarial robustness
def evaluate_arrays(yt,prob):
    yp=prob.argmax(1)
    return {
        "Accuracy":accuracy_score(yt,yp),
        "Precision":precision_score(yt,yp,average=avg,zero_division=0),
        "Recall":recall_score(yt,yp,average=avg,zero_division=0),
        "F1":f1_score(yt,yp,average=avg,zero_division=0)
    }

adv_preds=[]; adv_probs=[]; clean_preds=[]; clean_probs=[]; ys=[]
model.eval()
for xb,yb in test_loader:
    xb,yb=xb.to(device),yb.to(device)
    clean=model(xb)
    adv,_,_=evolutionary_adversarial(model,xb,yb)
    al=model(adv)
    clean_probs.append(F.softmax(clean,1).detach().cpu().numpy())
    adv_probs.append(F.softmax(al,1).detach().cpu().numpy())
    ys.append(yb.cpu().numpy())

yt=np.concatenate(ys); cp=np.concatenate(clean_probs); ap=np.concatenate(adv_probs)
clean=evaluate_arrays(yt,cp); adversarial=evaluate_arrays(yt,ap)
attack_success=(cp.argmax(1)!=ap.argmax(1)).mean()
retention=adversarial["F1"]/max(clean["F1"],1e-8)*100
robust=pd.DataFrame([{
    "Clean Accuracy (%)":clean["Accuracy"]*100,
    "Adversarial Accuracy (%)":adversarial["Accuracy"]*100,
    "Clean F1 (%)":clean["F1"]*100,
    "Adversarial F1 (%)":adversarial["F1"]*100,
    "Attack Success Rate (%)":attack_success*100,
    "Robustness Retention (%)":retention
}])
display(robust)
robust.to_excel(os.path.join(SAVE_DIR,"adversarial_robustness.xlsx"),index=False)


## 20. Five-fold accuracy analysis

The paper requests 5-fold comparison. For the proposed implementation, this cell runs five stratified folds using the same preprocessing recipe and a compact training schedule. Increase `KFOLD_EPOCHS` for final experiments.

In [ ]:
# 20. Five-fold evaluation of the proposed model
from sklearn.model_selection import StratifiedKFold
KFOLD_EPOCHS=2
# Use a bounded subset for practical execution; set KFOLD_MAX=None for the full dataset.
KFOLD_MAX=50000
Xk=X_train_s[:KFOLD_MAX] if KFOLD_MAX else X_train_s
yk=y_train[:KFOLD_MAX] if KFOLD_MAX else y_train

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
fold_rows=[]
for fold,(tri,vi) in enumerate(skf.split(Xk,yk),1):
    # simple sequence-compatible fold construction
    trX,trY=Xk[tri],yk[tri]
    vaX,vaY=Xk[vi],yk[vi]
    if len(trX)<SEQ_LEN or len(vaX)<SEQ_LEN: continue
    trXs,trYs=make_sequences(trX,trY,SEQ_LEN)
    vaXs,vaYs=make_sequences(vaX,vaY,SEQ_LEN)
    tl=DataLoader(TrafficDataset(trXs,trYs),batch_size=BATCH_SIZE,shuffle=True)
    vl=DataLoader(TrafficDataset(vaXs,vaYs),batch_size=BATCH_SIZE)
    m=FullModel(Xk.shape[1],NUM_CLASSES).to(device)
    o=torch.optim.AdamW(m.parameters(),lr=LR)
    for ep in range(KFOLD_EPOCHS):
        m.train()
        for xb,yb in tl:
            xb,yb=xb.to(device),yb.to(device); o.zero_grad()
            loss=criterion(m(xb),yb); loss.backward(); o.step()
    yy,pp,_=predict_loader(m,vl)
    fold_rows.append({"Fold":fold,"Accuracy (%)":accuracy_score(yy,pp)*100})
    print(f"Fold {fold}: {fold_rows[-1]['Accuracy (%)']:.3f}%")

fold_df=pd.DataFrame(fold_rows)
if len(fold_df):
    fold_df.loc[len(fold_df)]={"Fold":"Mean","Accuracy (%)":fold_df["Accuracy (%)"].mean()}
    fold_df.loc[len(fold_df)]={"Fold":"Std","Accuracy (%)":fold_df.iloc[:-1]["Accuracy (%)"].std(ddof=1)}
display(fold_df)
fold_df.to_excel(os.path.join(SAVE_DIR,"kfold_accuracy.xlsx"),index=False)


In [ ]:
# 21. Statistical significance analysis
from scipy.stats import ttest_rel

# Compare repeated fold accuracies where available.
# For a valid paired test, both methods need fold-level predictions.
# This cell compares the proposed model's five folds with user-supplied/reference fold arrays if available.
# Example:
# baseline_folds={"MambaCNN":[...],...}
baseline_folds={}
rows=[]
prop=np.array(fold_df.iloc[:-2]["Accuracy (%)"],dtype=float) if len(fold_df)>=3 else np.array([])
for name,vals in baseline_folds.items():
    vals=np.array(vals,dtype=float)
    if len(vals)==len(prop) and len(vals)>=2:
        stat,p=ttest_rel(prop,vals)
        rows.append({"Method":name,"Test Statistic":stat,"p-value":p,
                     "Significance":"Significant (p<0.05)" if p<0.05 else "Not Significant (p>=0.05)"})
stats_df=pd.DataFrame(rows)
display(stats_df)
stats_df.to_excel(os.path.join(SAVE_DIR,"statistical_significance.xlsx"),index=False)


## 22. Computational efficiency

The paper requests parameters, trainable parameters, FLOPs, training time, inference time, GPU memory and model size. fileciteturn0file0L853-L907

The proposed row below is measured from this implementation. Baseline rows should be generated by actually implementing/running the corresponding baseline models rather than entering assumed values.

In [ ]:
# 22. Computational measurements for the proposed model
import os, time
params=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)

model_path=os.path.join(SAVE_DIR,"model_state_tmp.pt")
torch.save(model.state_dict(),model_path)
size_mb=os.path.getsize(model_path)/1024**2
os.remove(model_path)

# Inference timing on one batch
model.eval()
xb,yb=next(iter(test_loader)); xb=xb.to(device)
if device.type=="cuda": torch.cuda.synchronize()
t0=time.perf_counter()
with torch.no_grad(): _=model(xb,hard=True)
if device.type=="cuda": torch.cuda.synchronize()
infer_ms=(time.perf_counter()-t0)/len(xb)*1000

gpu_mem=torch.cuda.max_memory_allocated()/1024**3 if device.type=="cuda" else np.nan
eff=pd.DataFrame([{
    "Method":"Proposed DepMamba-X + EvoAttack-X + RARE",
    "Parameters (M)":params/1e6,
    "Trainable Parameters (M)":trainable/1e6,
    "Inference Time (ms/sample)":infer_ms,
    "GPU Memory (GB)":gpu_mem,
    "Model Size (MB)":size_mb
}])
display(eff)
eff.to_excel(os.path.join(SAVE_DIR,"computational_efficiency_proposed.xlsx"),index=False)


In [ ]:
# 23. Save complete experiment metadata
metadata={
    "seed":SEED,"device":str(device),"sequence_length":SEQ_LEN,
    "selected_features":selected_features,"classes":classes,
    "num_classes":NUM_CLASSES,"hidden":HIDDEN,"embed_dim":EMBED_DIM,
    "num_experts":NUM_EXPERTS,"top_k":TOP_K,
    "pretrain_epochs":PRETRAIN_EPOCHS,"rare_epochs":RARE_EPOCHS,
    "attack_pool":ATTACK_POOL,"attack_steps":ATTACK_STEPS,
    "attack_eps":ATTACK_EPS,"attack_alpha":ATTACK_ALPHA
}
with open(os.path.join(SAVE_DIR,"experiment_config.json"),"w") as f: json.dump(metadata,f,indent=2)
print("All outputs saved under:",SAVE_DIR)


## 24. Reproducibility notes

- The paper's dataset description states that the data contain four IDS sources, normal/malicious traffic, PCAPNG/CSV representations, source/destination IPs, ports and TCP payload fields. fileciteturn0file0L245-L255
- The preprocessing follows robust scaling and mutual-information selection described in Section 3.4. fileciteturn0file0L285-L310
- DepMamba-X, EvoAttack-X, RARE, SAGE, U-CAGE, LATe and X-EASE are implemented as executable modules.
- Baseline comparison values are intentionally **not fabricated**. To reproduce the paper's Tables 4.2–4.9 rigorously, each baseline must be trained/evaluated on the same split and protocol.
- For a publication run, increase epochs, use the full dataset, document hardware, and save the exact train/validation/test split.